# AIDA 1141 Project Exercise 08 – Error and Slice Review

**Question:** Which validation assessments does the model get wrong or place near its threshold, what do those cases have in common, and does performance differ across operational slices?

This notebook uses the saved Exercise 06 validation predictions and threshold; nothing is refitted and test data is not used. The assessment-time inputs are added from Exercise 02 to describe each case.

Simulation-only fields (base processing days, workload delay, hidden delay, and delay reason) are added for error analysis only. They were never model inputs and are not used to change the model. They support post-prediction review of patterns associated with errors, but do not establish why individual predictions failed. All results describe methods applied to simulated outcomes, not real employees or HR performance.

In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

EX02 = Path("../02-synthetic-request-records")
EX06 = Path("../06-model-training-and-split")
EX07 = Path("../07-evaluation-measures")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

read = lambda name: pd.read_csv(EX02 / "data" / name, keep_default_na=False, na_values=[""])
predictions = pd.read_csv(EX06 / "outputs" / "validation_predictions.csv")
settings = pd.read_csv(EX06 / "outputs" / "model_settings.csv", index_col=0)["value"]
MODEL_THRESHOLD = float(settings["threshold"])

INPUTS = ["business_days_until_start", "current_status", "business_days_in_current_status", "team_open_workload"]
REQUEST_FIELDS = ["task_type", "hiring_department", "employee_type", "intake_channel", "info_issue_at_intake"]
SIMULATION_FIELDS = ["sim_base_processing_days", "sim_workload_delay_days", "sim_hidden_delay_days", "sim_delay_reason"]
validation = (predictions
              .merge(read("assessments.csv")[["assessment_id"] + INPUTS], on="assessment_id", how="left",
                     validate="one_to_one")
              .merge(read("requests.csv")[["request_id"] + REQUEST_FIELDS], on="request_id", how="left",
                     validate="many_to_one")
              .merge(read("simulation_truth.csv")[["request_id"] + SIMULATION_FIELDS], on="request_id", how="left",
                     validate="many_to_one"))
ex07_confusion = pd.read_csv(EX07 / "outputs" / "confusion_matrices_assessments.csv", index_col=0)
print(f"Loaded {len(validation)} validation assessments; model threshold {MODEL_THRESHOLD:.4f} (rounded for display)")

Loaded 526 validation assessments; model threshold 0.1254 (rounded for display)


## A. Consistency checks

These checks confirm that every validation assessment matched its inputs and simulation values, that the saved threshold reproduces the model's flags, and that the confusion counts match Exercise 07. Each early-warning assessment is then labelled TP, FP, FN, or TN for the model and for the global days rule.

In [4]:
MODEL, RULE = "logistic_regression", "days_until_start_rule"

# Validate the True/False columns and probabilities before using them
boolean_ok = all(validation[c].dtype == bool for c in ["early_warning", "at_risk", MODEL, RULE])
probability_ok = validation["risk_probability"].notna().all() and validation["risk_probability"].between(0, 1).all()
assert boolean_ok and probability_ok, "Labels, flags, or probabilities are missing or invalid."

early = validation[validation["early_warning"]].copy()

def outcome(actual, flagged):
    """TP, FP, FN, or TN for each assessment."""
    return np.select([flagged & actual, flagged & ~actual, ~flagged & actual], ["TP", "FP", "FN"], default="TN")

early["model_outcome"] = outcome(early["at_risk"], early[MODEL])
early["rule_outcome"] = outcome(early["at_risk"], early[RULE])
model_counts = early["model_outcome"].value_counts().reindex(["TP", "FP", "FN", "TN"])
rule_counts = early["rule_outcome"].value_counts().reindex(["TP", "FP", "FN", "TN"])

checks = pd.Series({
    "526 validation assessments, one row each": len(validation) == 526 and validation["assessment_id"].is_unique,
    "Every assessment matched its inputs and simulation values":
        validation[INPUTS + REQUEST_FIELDS + SIMULATION_FIELDS].notna().all().all(),
    "Label, early-warning, and flag columns are True/False only": boolean_ok,
    "Every probability is present and between 0 and 1": probability_ok,
    "Saved threshold reproduces the model's flags":
        ((validation["risk_probability"] >= MODEL_THRESHOLD) == validation[MODEL]).all(),
    "Confusion counts match Exercise 07":
        model_counts.tolist() == ex07_confusion.loc[MODEL, ["TP", "FP", "FN", "TN"]].tolist()
        and rule_counts.tolist() == ex07_confusion.loc[RULE, ["TP", "FP", "FN", "TN"]].tolist(),
}, name="passed")
assert checks.all(), "Inputs do not match Exercises 06 and 07."
checks.to_frame()

,passed
"526 validation assessments, one row each",True
Every assessment matched its inputs and simulation values,True
"Label, early-warning, and flag columns are True/False only",True
Every probability is present and between 0 and 1,True
Saved threshold reproduces the model's flags,True
Confusion counts match Exercise 07,True


## B. Error-review table

Representative validation early-warning assessments are selected from six groups:

1. **Missed at-risk:** every model false negative.
2. **Highest-scoring false positives:** the five not-at-risk assessments with the highest estimated risk.
3. **Boundary band, just above** and 4. **just below the threshold:** cases within a predefined score band around the decision threshold are selected for boundary review. The band runs from half to double the threshold. It is exploratory and does not represent a confidence interval or the final review-queue policy.
5. **Rule flagged, model did not, not at risk:** model–rule disagreements where the model avoided a false positive the rule made, choosing the lowest estimated risk.
6. **Correct contrast:** the two highest-scoring true positives.

Cases are deliberately selected as representative examples of each review group, to illustrate particular patterns. They are not a random or statistically representative sample of all assessments, and some assessments appear in more than one group. Ties in the sorting score are broken by assessment ID, so the selection is reproducible.

Each row shows the assessment-time inputs, the model's estimated risk and flag, the global rule's flag, the simulated outcome, and the simulation-only audit fields used for analysis.

In [5]:
band_low, band_high = MODEL_THRESHOLD / 2, MODEL_THRESHOLD * 2
early["in_boundary_band"] = early["risk_probability"].between(band_low, band_high, inclusive="left")
early["model_rule_disagree"] = early[MODEL] != early[RULE]

group_sizes = pd.Series({
    "Missed at-risk (model false negatives)": (early["model_outcome"] == "FN").sum(),
    "Model false positives": (early["model_outcome"] == "FP").sum(),
    f"Boundary-review band ({band_low:.3f} to {band_high:.3f})": early["in_boundary_band"].sum(),
    "Model and global rule disagree": early["model_rule_disagree"].sum(),
}, name="assessments")
disagreements = (early[early["model_rule_disagree"]].groupby(["at_risk", MODEL]).size()
                 .rename("assessments").reset_index()
                 .rename(columns={MODEL: "model_flagged"}).assign(rule_flagged=lambda d: ~d["model_flagged"]))

def pick(rows, group, n=None, by=None, ascending=False):
    """Select review cases from one group, optionally the top n by a column (ties broken by assessment ID)."""
    rows = rows.sort_values([by, "assessment_id"], ascending=[ascending, True]) if by else rows
    return (rows.head(n) if n else rows).assign(review_group=group)

band = early[early["in_boundary_band"]].assign(distance=lambda d: (d["risk_probability"] - MODEL_THRESHOLD).abs())
cases = pd.concat([
    pick(early[early["model_outcome"] == "FN"], "1 missed at-risk (all)", by="risk_probability"),
    pick(early[early["model_outcome"] == "FP"], "2 highest-scoring false positives", n=5, by="risk_probability"),
    pick(band[band[MODEL]], "3 boundary band, just above threshold", n=3, by="distance", ascending=True),
    pick(band[~band[MODEL]], "4 boundary band, just below threshold", n=3, by="distance", ascending=True),
    pick(early[early["model_rule_disagree"] & ~early[MODEL] & ~early["at_risk"]],
         "5 rule flagged, model did not, not at risk", n=3, by="risk_probability", ascending=True),
    pick(early[early["model_outcome"] == "TP"], "6 correct: highest-scoring true positives", n=2, by="risk_probability"),
])
COLUMNS = ["review_group", "assessment_id", "request_id", "checkpoint", "at_risk", "risk_probability", MODEL, RULE,
           "business_days_until_start", "current_status", "business_days_in_current_status", "team_open_workload",
           "task_type", "hiring_department", "employee_type", "intake_channel", "info_issue_at_intake",
           "sim_base_processing_days", "sim_hidden_delay_days", "sim_delay_reason", "sim_workload_delay_days"]
error_review = cases[COLUMNS].round(3).reset_index(drop=True)
error_review.to_csv(OUTPUT_DIR / "error_review_cases.csv", index=False)

display(group_sizes.to_frame())
display(disagreements)
error_review

,assessments
Missed at-risk (model false negatives),6
Model false positives,95
Boundary-review band (0.063 to 0.251),128
Model and global rule disagree,78


,at_risk,model_flagged,assessments,rule_flagged
0,False,False,63,True
1,False,True,9,False
2,True,False,4,True
3,True,True,2,False


,review_group,assessment_id,request_id,checkpoint,at_risk,risk_probability,logistic_regression,days_until_start_rule,business_days_until_start,current_status,...,team_open_workload,task_type,hiring_department,employee_type,intake_channel,info_issue_at_intake,sim_base_processing_days,sim_hidden_delay_days,sim_delay_reason,sim_workload_delay_days
0,1 missed at-risk (all),ASM-02388,REQ-01922,checkpoint_1,True,0.123,False,True,6,open,...,3,orientation_scheduling,Transit,temporary,email,False,2,3,staff_absence,0
1,1 missed at-risk (all),ASM-00191,REQ-00177,checkpoint_1,True,0.101,False,True,6,open,...,16,system_access,Parks,temporary,web_form,False,4,4,vendor_delay,0
2,1 missed at-risk (all),ASM-02557,REQ-02052,checkpoint_1,True,0.097,False,False,8,open,...,3,payroll_setup,Finance,permanent,web_form,False,3,6,staff_absence,0
3,1 missed at-risk (all),ASM-02569,REQ-02052,checkpoint_2,True,0.090,False,True,5,in_progress,...,3,payroll_setup,Finance,permanent,web_form,False,3,6,staff_absence,0
4,1 missed at-risk (all),ASM-00355,REQ-00263,checkpoint_2,True,0.044,False,True,5,in_progress,...,5,workplace_access,Recreation,permanent,web_form,False,5,5,backlog,0
5,1 missed at-risk (all),ASM-00285,REQ-00263,checkpoint_1,True,0.011,False,False,11,open,...,6,workplace_access,Recreation,permanent,web_form,False,5,5,backlog,0
6,2 highest-scoring false positives,ASM-02332,REQ-01875,checkpoint_1,False,0.732,True,True,4,open,...,13,payroll_setup,Recreation,temporary,web_form,False,2,0,none,0
7,2 highest-scoring false positives,ASM-01286,REQ-01045,checkpoint_2,False,0.691,True,True,5,open,...,43,equipment_setup,Parks,seasonal,phone,False,3,0,none,1
8,2 highest-scoring false positives,ASM-02186,REQ-01756,checkpoint_1,False,0.674,True,True,4,open,...,5,payroll_setup,Recreation,temporary,web_form,False,2,0,none,0
9,2 highest-scoring false positives,ASM-01280,REQ-01045,checkpoint_1,False,0.635,True,True,6,open,...,48,equipment_setup,Parks,seasonal,phone,False,3,0,none,1
